### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_aquatic_toxicity",
    dataset_year="2014",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5SG7H",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/505/qsar+aquatic+toxicity.zip && unzip qsar+aquatic+toxicity.zip && rm qsar+aquatic+toxicity.zip && mkdir -p local-data-warehouse/qsar_aquatic_toxicity && mv qsar_aquatic_toxicity.csv local-data-warehouse/qsar_aquatic_toxicity/
""",
    # References
    academic_reference_bibtex="""@article{cassotti2014prediction,
  title={Prediction of acute aquatic toxicity toward daphnia magna by using the ga-k nn method},
  author={Cassotti, Matteo and Ballabio, Davide and Consonni, Viviana and Mauri, Andrea and Tetko, Igor V and Todeschini, Roberto},
  journal={Alternatives to Laboratory Animals},
  volume={42},
  number={1},
  pages={31--41},
  year={2014},
  publisher={SAGE Publications Sage UK: London, England}
}
""",
    academic_reference_bibtex_key="cassotti2014prediction",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- Note, the dataset has only a subset of features of the original data. The original data was reduced through feature selection with kNN by the authors in the paper above. We only have the 8 descriptors available in the UCI version. This might leak target information into the feature selection (or behave like an expert giving us the best features). Moreover, it biases the datasets and might make it trivial.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LC50",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
columns = [
    "TPSA",
    "SAacc",
    "H-050",
    "MLOGP",
    "RDCHI",
    "GATS1p",
    "nN",
    "C-040",
    "LC50"
]
df = pd.read_csv(dataset_mold.path / "qsar_aquatic_toxicity.csv", sep=";", names=columns)
print("Loaded data shape:", df.shape)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()